# BERT を使用したテキスト分類

### 学習目標
1. KerasHub から事前学習済み BERT モデルを読み込む方法を学ぶ
2. 分類器と組み合わせて独自のモデルを構築する方法を学ぶ
3. ファインチューニングによって LLM（BERT）モデルをトレーニングする方法を学ぶ
4. トレーニング済みモデルを保存して使用する方法を学ぶ
5. テキスト分類モデルを評価する方法を学ぶ

このラボでは、プレーンテキストの IMDB 映画レビューのデータセットに対して感情分析を実行するために、BERT をファインチューニングする方法を学びます。
モデルのトレーニングに加えて、テキストを適切な形式に前処理する方法も学びます。

### 始める前に

トレーニングに時間がかかりすぎないように、Notebook インスタンスに GPU（1 x NVIDIA Tesla T4 で十分です）がアタッチされていることを確認してください。

## BERT について

[BERT](https://arxiv.org/abs/1810.04805) やその他の Transformer エンコーダ アーキテクチャは、NLP（自然言語処理）のさまざまなタスクで大きな成功を収めてきました。これらは、ディープ ラーニング モデルでの使用に適した自然言語のベクトル空間表現を計算します。BERT ファミリーのモデルは、Transformer エンコーダ アーキテクチャを使用して、入力テキストの各トークンを前後すべてのトークンの完全なコンテキスト内で処理します。これが「Bidirectional Encoder Representations from Transformers」（Transformer による双方向のエンコーダ表現）という名前の由来です。

BERT モデルは通常、大規模なテキスト コーパスで事前学習され、その後、特定のタスクに合わせてファインチューニングされます。

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"



In [ ]:
import datetime
import shutil

import keras
import keras_hub as hub
import matplotlib.pyplot as plt
import tensorflow as tf
from google.cloud import aiplatform

tf.get_logger().setLevel("ERROR")

GPU が接続されているか確認するには、以下を実行します。

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

### 感情分析

このノートブックでは、レビューのテキストに基づいて映画レビューを *positive*（肯定的）または *negative*（否定的）に分類する感情分析モデルをトレーニングします。

[Internet Movie Database](https://www.imdb.com/) の 50,000 件の映画レビューのテキストを含む [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/) を使用します。

### IMDB データセットのダウンロード

データセットをダウンロードして展開し、ディレクトリ構造を確認しましょう。

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

path = "./bert_dataset/"
os.makedirs(path, exist_ok=True) 

dataset = keras.utils.get_file(
    "aclImdb_v1.tar.gz", url, untar=True, cache_dir=path, cache_subdir=""
)

dataset_dir = os.path.join(
    os.path.dirname(dataset), "aclImdb_v1_extracted/aclImdb"
)

train_dir = os.path.join(dataset_dir, "train")

# データの読み込みを容易にするため、未使用のフォルダを削除
remove_dir = os.path.join(train_dir, "unsup")
shutil.rmtree(remove_dir)

次に、`text_dataset_from_directory` ユーティリティを使用して、ラベル付きの `tf.data.Dataset` を作成します。

IMDB データセットはすでにトレーニング用とテスト用に分割されていますが、検証セットがありません。以下の `validation_split` 引数を使用して、トレーニング データを 80:20 の比率で分割し、検証セットを作成しましょう。

注: `validation_split` 引数と `subset` 引数を使用する場合は、検証分割とトレーニング分割が重複しないように、ランダム シードを指定するか、`shuffle=False` を渡してください。

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 42

raw_train_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=seed,
)

class_names = raw_train_ds.class_names
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)

val_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/train",
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=seed,
)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = keras.utils.text_dataset_from_directory(
    path + "aclImdb_v1_extracted/aclImdb/test", batch_size=batch_size
)

test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

いくつかのレビューを見てみましょう。

In [ ]:
for text_batch, label_batch in train_ds.take(1):
    for i in range(3):
        print(f"Review: {text_batch.numpy()[i]}")
        label = label_batch.numpy()[i]
        print(f"Label : {label} ({class_names[label]})")

## KerasHub からのモデルの読み込み

このラボでは、Small BERT と呼ばれるモデルを読み込みます。Small BERT は元の BERT と同じ一般的なアーキテクチャを備えていますが、Transformer ブロックの数が少ないか、サイズが小さくなっています。

その他の代表的な BERT モデルには、BERT Base、ALBERT、BERT Experts、Electra などがあります。詳細については、このラボの最後にある学習の継続セクションを参照してください。

以下で利用可能なモデルのほかに、より大規模でさらに高い精度が得られるモデルの[複数のバージョン](https://keras.io/keras_hub/presets/)がありますが、単一の GPU でファインチューニングするには大きすぎます。

以下のコードでは、モデル間のすべての違いがカプセル化されているため、モデル名を切り替えるだけでこれらの任意のモデルを試すことができます。

### ファインチューニングする BERT モデルの選択

In [ ]:
bert_encoder = hub.models.BertBackbone.from_preset(
    "bert_small_en_uncased",
    load_weights=True,
)

print(
    "*******************************\nPre-Trained Model Architecture\n*******************************"
)
bert_encoder.summary()

## 前処理モデル

テキスト入力を BERT に入力する前に、数値のトークン ID に変換し、いくつかのテンソルに配置する必要があります。KerasHub は、上記で説明した各 BERT モデルに対応する前処理モデルを提供しており、この変換を実装しています。テキストを前処理するために、Keras モデルの外部で純粋な Python コードを実行する必要はありません。

前処理モデルは、後でモデル内でファインチューニングするモデルと同じである必要があります。

注: ファインチューニングされたモデルを構成するために、[hub.models.BertTextClassifierPreprocessor](https://keras.io/keras_hub/api/models/bert/bert_text_classifier_preprocessor/) を使用して前処理モデルを読み込みます。

In [ ]:
bert_preprocessor = hub.models.BertPreprocessor.from_preset(
    "bert_small_en_uncased", name="preprocessor", truncate="waterfall"
)

いくつかのテキストで前処理モデルを試して、出力を確認してみましょう。

前処理モデル関数を呼び出し、`text_test` を渡します。

In [ ]:
text_test = ["this is such an amazing movie!"]
text_preprocessed = bert_preprocessor(text_test)

# この出力ボックスは前処理された辞書のキーを確認するのに役立ちます
print(f"Keys       : {list(text_preprocessed.keys())}")

# 1. token_ids はトークン化された文内の単語の ID
print(f'Shape      : {text_preprocessed["token_ids"].shape}')
print(f'Token Ids   : {text_preprocessed["token_ids"][0, :12]}')

# 2. padding_mask はマスクするトークン（マスク言語モデル）
print(f'Padding Mask : {text_preprocessed["padding_mask"][0, :12]}')

# 3. segment_ids は入力文の文 ID
print(f'Type Ids   : {text_preprocessed["segment_ids"][0, :12]}')

ご覧のとおり、BERT モデルが使用する前処理からの 3 つの出力（`token_ids`、`padding_mask`、`segment_ids`（または `type_ids`））が得られました。

その他の重要なポイント:
- 入力は 512 トークンに切り詰められます。
- 単一の文の入力であるため、`segment_ids` は 1 つの値（0）のみを持ちます。複数文の入力の場合、各入力に対応する番号が設定されます。

テキスト プリプロセッサは `keras.layers.Layer` オブジェクトのサブクラスです。つまり、個別に前処理を行う代わりに、モデル コード内のレイヤとして組み込むことができます。

### BERT モデルの使用

BERT を独自のモデルに組み込む前に、その出力を確認してみましょう。

In [ ]:
bert_encoder = hub.models.BertBackbone.from_preset(
    "bert_small_en_uncased", load_weights=True, name="bert_encoder"
)
bert_encoder.trainable = True

In [ ]:
bert_results = bert_encoder(text_preprocessed)

print(f'Pooled Outputs Shape:{bert_results["pooled_output"].shape}')
print(f'Pooled Outputs Values:{bert_results["pooled_output"][0, :12]}')
print(f'Sequence Outputs Shape:{bert_results["sequence_output"].shape}')
print(f'Sequence Outputs Values:{bert_results["sequence_output"][0, :12]}')

BERT モデルは、`pooled_output`、`sequence_output`、`encoder_outputs` という 3 つの重要なキーを持つマップを返します。

- `pooled_output`: 各入力シーケンス全体を表します。形状は `[batch_size, H]` です。これは映画レビュー全体の埋め込みと考えることができます。
- `sequence_output`: コンテキスト内の各入力トークンを表します。形状は `[batch_size, seq_length, H]` です。これは映画レビュー内のすべてのトークンに対するコンテキスト埋め込みと考えることができます。

ファインチューニングでは、`pooled_output` 配列を使用します。

### モデルの定義

前処理モデル、選択した BERT モデル、1 つの Dense レイヤ、Dropout レイヤを備えた、非常にシンプルなファインチューニング モデルを作成します。

注: ベースモデルの入出力に関する詳細情報については、モデルの URL からドキュメント ページを参照してください。

モデル内のレイヤの順序は次のようになります。
1. 入力レイヤ（Input Layer）
2. 前処理レイヤ（Pre-processing Layer）
3. エンコーダ レイヤ（Encoder Layer）
4. BERT 出力マップから `pooled_output` を使用
5. Dropout レイヤ
6. sigmoid 活性化関数を持つ Dense レイヤ

In [ ]:
def build_classifier_model(dropout_rate=0.1):
    # 1. 生テキスト用の入力レイヤを定義
    text_input = keras.layers.Input(shape=(), dtype="string", name="text")

    # 2. 入力テキストにプリプロセッサを適用
    # 'preprocessor' 変数は次の出力辞書を保持します:
    # {'token_ids', 'segment_ids', 'padding_mask'}
    preprocessor = bert_preprocessor(text_input)

    # 3. BERT バックボーン / エンコーダをロード
    encoder_outputs = bert_encoder(preprocessor)

    # 4. 文レベルの分類に 'pooled_output' を使用
    net = encoder_outputs["pooled_output"]

    # 5. 使用したいその他のレイヤを追加
    net = keras.layers.Dropout(dropout_rate, name="dropout")(net)
    net = keras.layers.Dense(1, activation="sigmoid", name="classifier")(net)
    return keras.Model(text_input, net)


dropout_rate = 0.15
classifier_model = build_classifier_model(dropout_rate)
bert_raw_result = classifier_model(keras.ops.convert_to_tensor(text_test))
print(bert_raw_result)

もちろん、モデルはまだトレーニングされていないため、出力には意味がありません。

モデルの構造を確認してみましょう。

In [ ]:
keras.utils.plot_model(classifier_model, show_layer_names=True, dpi=100)

## モデルのトレーニング

前処理モジュール、BERT エンコーダ、データ、分類器など、モデルをトレーニングするためのすべてのパーツが揃いました。

### 損失関数

これは二値分類問題であり、モデルは確率を出力する（単一ユニットのレイヤ）ため、`losses.BinaryCrossentropy` 損失関数を使用します。

ここで損失関数と評価指標を定義します。二値分類であるため、BinaryCrossentropy と BinaryAccuracy を使用します。

In [ ]:
loss = keras.losses.BinaryCrossentropy()
metrics = keras.metrics.BinaryAccuracy()

### オプティマイザ

ファインチューニングには、BERT が元々トレーニングされていたのと同じオプティマイザである「Adaptive Moments」（Adam）を使用しましょう。このオプティマイザは予測損失を最小限に抑え、重み減衰（モーメントを使用しない）によって正則化を行います。これは [AdamW](https://keras.io/api/optimizers/adamw/) としても知られています。AdamW 最適化は、1 次および 2 次モーメントの適応的推定に基づき、重みを減衰させる手法を追加した確率的勾配降下法です。

これまでのラボでは一般的な選択肢である Adam オプティマイザを使用してきました。しかし、このラボでは重み減衰を使用して汎化性能を向上させることを目的とした、Adam の変形である AdamW を使用します。AdamW の背後にある直感とアルゴリズムは、こちらの[論文](https://arxiv.org/abs/1711.05101)で確認できます。

学習率（`learning_rate`）については、BERT の論文に沿って、ファインチューニングでは初期学習率を小さく設定します（5e-5、3e-5、2e-5 のいずれかが最適）。

In [ ]:
epochs = 5

optimizer = keras.optimizers.AdamW(learning_rate=3e-5, weight_decay=0.004)

### BERT モデルの読み込みとトレーニング

先ほど作成した `classifier_model` を使用して、損失、評価指標、オプティマイザでモデルをコンパイルできます。

上で定義したオプティマイザ、損失、評価指標を使用してモデルをコンパイルします。

*注: GPU との互換性の問題により、ここでは `jit_compile` という属性を `False` に設定しています*

In [ ]:
classifier_model.compile(
    optimizer=optimizer, loss=loss, metrics=[metrics], jit_compile=False
)

注: トレーニング時間は、選択した BERT モデルの複雑さによって異なります。

モデルを数エポック分トレーニングしましょう。1 基の NVIDIA T4 GPU を使用した場合、2 エポックのトレーニングには約 15〜20 分かかることに注意してください。

In [ ]:
print(f"Training model with BERT")
history = classifier_model.fit(
    x=train_ds, validation_data=val_ds, epochs=epochs
)

### モデルの評価

モデルのパフォーマンスを確認してみましょう。損失（誤差を表す数値、低いほど良い）と精度の 2 つの値が返されます。

In [ ]:
loss, accuracy = classifier_model.evaluate(test_ds)

print(f"Loss: {loss}")
print(f"Accuracy: {accuracy}")

### 精度と損失の推移のプロット

`model.fit()` から返された `History` オブジェクトに基づいて、比較のためにトレーニングと検証の損失、およびトレーニングと検証の精度をプロットできます。

In [ ]:
history_dict = history.history
print(history_dict.keys())

acc = history_dict["binary_accuracy"]
val_acc = history_dict["val_binary_accuracy"]
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]

epochs = range(1, len(acc) + 1)
fig = plt.figure(figsize=(10, 6))
fig.tight_layout()

plt.subplot(2, 1, 1)
# "bo" は「青い点」を表します
plt.plot(epochs, loss, "r", label="Training loss")
# "b" は「青い実線」を表します
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
# plt.xlabel('Epochs')
plt.ylabel("Loss")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(epochs, acc, "r", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")

このプロットでは、赤い線がトレーニングの損失と精度を表し、青い線が検証の損失と精度を表します。

### 推論用のエクスポート

後で使用できるように、ファインチューニングされたモデルを保存します。

推論用にモデルを保存するには、`SAVEDMODEL_PATH` で指定されたパスにエクスポートします。Keras 3 ではモデルを保存するためのメソッドが提供されており、このノートブックでは [model.export()](https://keras.io/api/models/model_saving_apis/export/) を使用しています。この関数を選択したのは、サービング専用に設計され、Google Cloud の Agent Platform へのデプロイと互換性がある TensorFlow SavedModel 形式でモデルを生成するためです。

In [ ]:
dataset_name = "imdb"
saved_model_path = "./{}_bert".format(dataset_name.replace("/", "_"))
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

EXPORT_PATH = os.path.join(saved_model_path, TIMESTAMP)


SAVEDMODEL_PATH = os.path.join(EXPORT_PATH, "savedmodel")

classifier_model.export(SAVEDMODEL_PATH)

メモリ内に残っているモデルと並べて試すことができるように、モデルを再読み込みしてみましょう。モデルを TF SavedModel 形式として保存したため、`tf.saved_model.load()` を使用して再読み込みします。

In [ ]:
reloaded_model = tf.saved_model.load(SAVEDMODEL_PATH)

ここでは任意の文でモデルをテストできます。下の `examples` 変数に追加するだけです。

In [ ]:
def print_my_examples(inputs, results):
    result_for_printing = [
        f"input: {inputs[i]:<30} : score: {results[i][0]:.6f}"
        for i in range(len(inputs))
    ]
    print(*result_for_printing, sep="\n")
    print()


examples = [
    "this is such an amazing movie!",  # これは先ほど試したのと同じ文です
    "The movie was great!",
    "The movie was meh.",
    "The movie was okish.",
    "The movie was terrible...",
]

reloaded_results = reloaded_model.serve(keras.ops.convert_to_tensor(examples))
original_results = classifier_model(keras.ops.convert_to_tensor(examples))

print("Results from the saved model:")
print_my_examples(examples, reloaded_results)
print("Results from the model in memory:")
print_my_examples(examples, original_results)

## （オプション）オンライン予測を取得するために Agent Platform にモデルをデプロイする

モデルを TensorFlow SavedModel 形式にエクスポートしました。この形式のモデルがあれば、ウェブ アプリケーション、JavaScript、モバイル アプリケーションなどからモデルを「サービング」する多くの方法があります。

次に、SavedModel コマンドライン インターフェース コマンド `saved_model_cli` を使用して、保存したモデルのシグネチャを出力します。コマンドライン インターフェースおよびサポートされている `show` や `run` コマンドの詳細については、[こちらのドキュメント](https://www.tensorflow.org/guide/saved_model#overview_of_commands)を参照してください。

[TF Serving](https://www.tensorflow.org/tfx/guide/serving) でモデルを使用したい場合は、名前付きシグネチャのいずれかを介して SavedModel が呼び出されることに留意してください。Python では、次のようにテストできます。

In [ ]:
!saved_model_cli show \
    --tag_set serve \
    --signature_def serving_default \
    --dir {SAVEDMODEL_PATH}

!find {SAVEDMODEL_PATH}
os.environ['SAVEDMODEL_PATH'] = SAVEDMODEL_PATH

In [ ]:
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
PROJECT = !gcloud config get-value project 2>/dev/null
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"
MODEL_DISPLAYNAME = f"classification-bert-{TIMESTAMP}"

print(f"MODEL_DISPLAYNAME: {MODEL_DISPLAYNAME}")

# from https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/predictions/pre-built-containers
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"
)

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
# GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gcloud storage ls
fi

In [ ]:
!gcloud storage cp -r $SAVEDMODEL_PATH gs://$BUCKET/$MODEL_DISPLAYNAME

モデルを Agent Platform のモデル レジストリにアップロードします。

In [ ]:
uploaded_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAYNAME,
    artifact_uri=f"gs://{BUCKET}/{MODEL_DISPLAYNAME}",
    serving_container_image_uri=SERVING_CONTAINER_IMAGE_URI,
)

エンドポイントを作成し、作成されたエンドポイントにモデルをデプロイします。

In [ ]:
MACHINE_TYPE = "n1-standard-4"

endpoint = uploaded_model.deploy(
    machine_type=MACHINE_TYPE,
    accelerator_type=None,
    accelerator_count=None,
)

モデルがエンドポイントにアップロードされたら、エンドポイントにクエリを送信してモデルから予測を取得できます。クエリはインスタンスのリストである必要があります。モデルのシグネチャから、クエリのキーは `text` でなければならないことがわかります。

In [ ]:
instances = [
    {"text": ["I loved the movie and highly recomment it"]},
    {"text": ["It was an okay movie in my opinion"]},
    {"text": ["I hated the movie"]},
]

In [ ]:
response = endpoint.predict(instances=instances)

In [ ]:
print(" prediction:", response.predictions)

## クリーンアップ
オンライン予測のためにモデルをエンドポイントにデプロイする場合、最小の `min-replica-count` は 1 であり、ノード時間ごとに課金されます。そのため、不要な課金を減らすためにエンドポイントを削除しましょう。エンドポイントを削除する前に、まず接続されているすべてのモデルのデプロイを解除（undeploy）します。

In [ ]:
endpoint.undeploy_all()

...その後、エンドポイントを削除します。

In [ ]:
endpoint.delete()

#### License

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License